Author: Krish

In [1]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from datetime import date

In [2]:
data_path = "C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Spring 2025\\STAT390\\LegalAid\\Data\\CAR_-_EP_Flow_Activity_Queue__Agent_Names\\"
adhoc_data_path = 'C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Fall 2025\\LegalAid\\Adhoc\\Adhoc datasets\\'

In [3]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
print("Data files read = ",i)

Data files read =  53


In [4]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [5]:
def custdata(id):
    return df_main.loc[df_main['Contact Session ID'] == id,:]

In [6]:
df_main.shape

(3328626, 8)

In [7]:
df_main.dtypes

Contact Session ID                  object
EP Name                             object
Flow Name                           object
Activity Name                       object
Activity Start Timestamp    datetime64[ns]
Queue Name                          object
Agent Name                          object
Termination Reason                  object
dtype: object

In [8]:
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour

In [9]:
df_main.sort_values(by = ['Contact Session ID', 'Activity Start Timestamp'], inplace=True)

In [10]:
df_main.reset_index(inplace = True, drop = True)

In [11]:
df_main['Date'] = df_main['Activity Start Timestamp'].dt.date

In [12]:
df = df_main.copy()

In [13]:
# Example target activity names
target_activities = ["PreQueueMessage1", "PreQueueMessage2", "QueueMenu1", "PlayMOH300s"]

# Assuming your DataFrame is named df and has at least these columns:
# "Call ID" (or whatever uniquely identifies a call) and "Activity Name"

# Step 1: Find all call IDs that hit one of the target activities
call_ids_to_keep = df.loc[df["Activity Name"].isin(target_activities), "Contact Session ID"].unique()

# Step 2: Filter to keep *all* rows of those calls
filtered_df = df[df["Contact Session ID"].isin(call_ids_to_keep)].copy().reset_index(drop = True)

In [14]:
# Uses Queue Name
# create queue_name_df similar to menus_df but using 'Queue Name' instead of 'Activity Name'
results = []

for contact_id in filtered_df['Contact Session ID'].unique():
    temp = filtered_df.loc[filtered_df['Contact Session ID'] == contact_id].copy()
    temp = temp.sort_values('Activity Start Timestamp').reset_index(drop=True)
    
    for i in range(1, len(temp)):
        current_row = temp.iloc[i]
        prev_row = temp.iloc[i - 1]
        queue_name = current_row['Queue Name']
        if queue_name not in [np.nan, '']:
            
            results.append({'Contact Session ID': contact_id, 'Queue Name': queue_name})
            break
queue_name_df = pd.DataFrame(results)   

In [15]:
filtered_df['Activity Start Timestamp'] = pd.to_datetime(filtered_df['Activity Start Timestamp'])

In [50]:
# For only large queue sizes find the time spent in each queue
large_queue_df = queue_name_df[queue_name_df['Queue Name'].isin(
    queue_name_df['Queue Name'].value_counts()[lambda x: x > 500].index
)]

# go through the filtered df and find the time spent in each queue for each contact session id
time_results = []

for contact_id in large_queue_df['Contact Session ID'].unique():
    temp = filtered_df.loc[filtered_df['Contact Session ID'] == contact_id].copy()
    temp = temp.sort_values('Activity Start Timestamp').reset_index(drop=True)
    
    for i in range(1, len(temp)):
        current_row = temp.iloc[i]
        prev_row = temp.iloc[i - 1]
        queue_name = current_row['Queue Name']
        if queue_name in large_queue_df['Queue Name'].values:
            # Calculate occurences of PlayMOH300s and QueueMenu1 in queue
            # Start if PreQueue in Activity Name
            # Count occurences of PlayMOH300s and QueueMenu1 in Activity Name
            # In order to find the time in queue, we take the time from the occurence of "PreQueue" that occurs after if we find the queue name and take it until the last occurence of "PlayMOH300s" or "QueueMenu1"
            time_in_queue = 0
            prequeue_indices = temp.index[temp['Activity Name'].str.contains('PreQueue', case=False, na=False)].tolist()
            end_queue_indices = temp.index[temp['Activity Name'].str.contains('PlayMOH300s|QueueMenu1', case=False, na=False)].tolist()
            if prequeue_indices and end_queue_indices:
                
                time_in_queue = temp.loc[end_queue_indices[-1], 'Activity Start Timestamp'] - temp.loc[prequeue_indices[0], 'Activity Start Timestamp']
                time_in_queue = time_in_queue.total_seconds()
            date = temp.loc[prequeue_indices[0],'Activity Start Timestamp'].date()
            moh_count = temp['Activity Name'].str.contains('PlayMOH300s', case=False, na=False).sum()
            menu_count = temp['Activity Name'].str.contains('QueueMenu1', case=False, na=False).sum()
            total_count = moh_count + menu_count
            time_results.append({'Contact Session ID': contact_id, 'Queue Name': queue_name, 'Time in Queue (s)': time_in_queue, 
                                 'Time Count': total_count, 'date':date})
            break


In [29]:
temp = filtered_df.loc[filtered_df['Contact Session ID'] == '0019d572-7f3e-4380-bc1f-274fc63edcb6'].copy()
temp = temp.sort_values('Activity Start Timestamp').reset_index(drop=True)

In [46]:
temp.loc[prequeue_indices[0],'Activity Start Timestamp'].date()

datetime.date(2025, 9, 17)

In [47]:
for i in range(1, len(temp)):
    current_row = temp.iloc[i]
    prev_row = temp.iloc[i - 1]
    queue_name = current_row['Queue Name']
    if queue_name in large_queue_df['Queue Name'].values:
        # Calculate occurences of PlayMOH300s and QueueMenu1 in queue
        # Start if PreQueue in Activity Name
        # Count occurences of PlayMOH300s and QueueMenu1 in Activity Name
        # In order to find the time in queue, we take the time from the occurence of "PreQueue" that occurs after if we find the queue name and take it until the last occurence of "PlayMOH300s" or "QueueMenu1"
        time_in_queue = 0
        prequeue_indices = temp.index[temp['Activity Name'].str.contains('PreQueue', case=False, na=False)].tolist()
        end_queue_indices = temp.index[temp['Activity Name'].str.contains('PlayMOH300s|QueueMenu1', case=False, na=False)].tolist()
        if prequeue_indices and end_queue_indices:
            
            time_in_queue = temp.loc[end_queue_indices[-1], 'Activity Start Timestamp'] - temp.loc[prequeue_indices[0], 'Activity Start Timestamp']
            time_in_queue = time_in_queue.total_seconds()
        date = temp.loc[prequeue_indices[0],'Activity Start Timestamp'].date()
        moh_count = temp['Activity Name'].str.contains('PlayMOH300s', case=False, na=False).sum()
        menu_count = temp['Activity Name'].str.contains('QueueMenu1', case=False, na=False).sum()
        total_count = moh_count + menu_count
        time_results.append({'Contact Session ID': contact_id, 'Queue Name': queue_name, 'Time in Queue (s)': time_in_queue, 'Time Count': total_count,
                     'date': date})
        break

In [51]:
time_results_df = pd.DataFrame(time_results)
time_results_df.head()

,Contact Session ID,Queue Name,Time in Queue (s),Time Count,date
0,000cce29-1c76-4991-9445-bb5f9028f47d,SubSenior Consumer,371.0,2,2024-05-29
1,00116654-66ea-4582-a109-aacde13a620d,Consumer,370.0,2,2024-07-25
2,001741db-98b4-409f-8646-32ed0ecf6331,Benefits,371.0,2,2025-07-23
3,0018380f-cf12-40f5-b525-af190304c42b,SubSenior Benefits,699.0,4,2024-08-20
4,0019d572-7f3e-4380-bc1f-274fc63edcb6,SubSenior Tenant,0.0,0,2025-09-17


In [52]:
time_results_df['Time in Queue (s)'] = time_results_df['Time in Queue (s)']/60
time_results_df.rename(columns = {'Time in Queue (s)':'Time in Queue'}, inplace = True)

In [53]:
time_results_df.head()

,Contact Session ID,Queue Name,Time in Queue,Time Count,date
0,000cce29-1c76-4991-9445-bb5f9028f47d,SubSenior Consumer,6.183333,2,2024-05-29
1,00116654-66ea-4582-a109-aacde13a620d,Consumer,6.166667,2,2024-07-25
2,001741db-98b4-409f-8646-32ed0ecf6331,Benefits,6.183333,2,2025-07-23
3,0018380f-cf12-40f5-b525-af190304c42b,SubSenior Benefits,11.650000,4,2024-08-20
4,0019d572-7f3e-4380-bc1f-274fc63edcb6,SubSenior Tenant,0.000000,0,2025-09-17


In [24]:
time_results_df['Queue Name'].value_counts()

Queue Name
Family                1936
Consumer              1446
SubSenior Other       1187
Housing               1029
Benefits               785
SubSenior Tenant       683
SubSenior Benefits     682
Employment             510
SubSenior Consumer     507
Name: count, dtype: int64

In [54]:
time_results_df.to_csv(adhoc_data_path + 'queue_time_analysis.csv', index=False)

In [55]:
time_results_df.date.min()

datetime.date(2024, 4, 8)

In [56]:
time_results_df.date.max()

datetime.date(2025, 9, 30)